# Лекција 13 - Мемоари агента са Цогнее графовима знања


## Подешавање

Овај бележник демонстрира како направити интелигентног **асистента за кодирање** са сталном меморијом користећи [**Cognee**](https://www.cognee.ai/) графове знања и **Microsoft Agent Framework** (MAF).

Cognee претвара неструктурисани текст у структурисани, претраживи граф знања заснован на векторским уградњама — дајући вашем агенту богату, везу-свесну дугорочну меморију.

### Шта ћете научити
1. **Прављење графова знања**: Претворите профиле програмера и добре праксе у структурисано, претраживо знање.
2. **Интеграција Cognee са MAF**: Користите `@tool` функције да омогућите MAF агенту да претражује Cognee граф знања.
3. **Разговори осетљиви на сесију**: Одржавајте контекст кроз више питања у истој сесији.
4. **Дугорочна меморија**: Сачувајте важно знање кроз сесије и враћајте га у новим разговорима.

### Претпоставке
- Python 3.9+
- Redis који ради локално (`docker run -d -p 6379:6379 redis`) за управљање сесијама
- Кључ за LLM API (нпр. OpenAI) — подесите `LLM_API_KEY` у `.env`
- `CACHING=true` у `.env` (потребно за Cognee сесије)
- Azure AI Foundry пројекат са развијеним моделом за разговор
- `AZURE_AI_PROJECT_ENDPOINT` и `AZURE_AI_MODEL_DEPLOYMENT_NAME` у `.env`
- Azure CLI аутентификован (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Типови меморије агента

Овај белешник истражује иста три типа меморије из главног белешника Лекције 13, али користи Cognee као бекенд дугорочне меморије:

| Тип меморије | Механизам | Век трајања |
|---|---|---|
| **Радна** | `agent.create_session()` (MAF) | Један разговор |
| **Краткорочна** | Cognee session cache (Redis) | Једна сесија |
| **Дугорочна** | Cognee knowledge graph + vectors | Стална |

### Архитектура меморије Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Припремите Цогнее Складиште


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Deo 1 — Izgradnja baze znanja

Unosimo tri tipa podataka kako bismo kreirali sveobuhvatnu bazu znanja za našeg asistenta za kodiranje:

1. **Profil programera** — lična stručnost i tehničko iskustvo  
2. **Najbolje prakse Pythona** — Zen Pythona sa praktičnim smernicama  
3. **Istorijski razgovori** — prošle sesije pitanja i odgovora između programera i AI asistenata


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Визуелизујте граф знања

Cognee може да прикаже интерактивну HTML визуализацију ентитета и односа које је издвојио.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Обогатите меморију помоћу Memify

`memify()` анализира граф знања и генерише интелигентна правила — идентификујући обрасце, најбоље праксе и везе између појмова.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Део 2 — MAF агент са Cognee алатима

Сада креирамо MAF агента који може да упитује знанствену мрежу Cognee преко `@tool` функција. Ово омогућава агенту да искористи пуну снагу семантичке претраге осетљиве на структуру графа уз одржавање контекста разговора кроз сесије.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Радна меморија са сесијама

`AgentSession` (направљен преко `agent.create_session()`) пружа радну меморију у оквиру сесије. агент може да се позове на раније поруке, а истовремено и да упитује долгочасовни граф знања Cognee-а.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Нова сесија — дугорочно памћење опстаје

Покретање нове сесије брише радну меморију, али когнитивни граф знања Cognee и даље је доступан. Агенат може да врати исто дугорочно знање у потпуно новом разговору.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резиме

У овом бележнику сте направили асистента за програмирање који комбинује **MAF радну меморију** (`agent.create_session()`) са **Cognee графом знања дугорочног памћења**.

### Шта сте научили
1. **Конструкција графа знања**: Cognee прерађује неструктурирани текст и прави граф + векторску меморију.
2. **Обогаћивање графа помоћу memify**: Изведене чињенице и богатији односи преко вашег постојећег графа.
3. **Интеграција MAF + Cognee**: `@tool` функције омогућавају MAF агенту да природно упитује Cognee граф.
4. **Радна меморија + меморија дугорочног трајања**: `AgentSession` (путем `agent.create_session()`) пружа контекст сесије док Cognee пружа упорну базу знања.
5. **Филтрирано претраживање помоћу NodeSets**: Циљано претраживање специфичних подскупова графа знања (нпр. само принципи).

### Главне поуке
- **Cognee** претвара сиров текст у структурирану, односно паметну меморију која разуме везе — моћније од обичне векторске базе.
- **`@tool` функције** јасно повезују MAF агенте и спољне системе знања.
- **`AgentSession`** (путем `agent.create_session()`) одваја контекст по разговору од дуготрајног знања.
- Исти граф знања служи за више сесија и агената.

### Примена у стварном свету
- **Кодерски копилоти**: Преглед кода, анализа инцидената, помоћ у архитектури
- **Копилоти за корисничку подршку**: Асистенти за подршку на основу производа, често постављаних питања и CRM белешки
- **Интерни експертски копилоти**: Помоћници за политику, правно и безбедносно разумевање смерница
- **Уједињени слојеви података**: Комбинација структурираних и неструктурираних података у један граф за упитивање

### Следећи кораци
- Експериментишите са временском свешћу у Cognee
- Дефинишите OWL онтологију за квалитет доменски специфичног графа
- Додајте повратне информације корисника за боље претраживање током времена
- Масовно примењујте на више агената који деле исти слој меморије Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:  
Овај документ је преведен коришћењем АИ сервиса за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да превод буде прецизан, имајте у виду да аутоматски преводи могу да садрже грешке или нетачности. Оригинални документ на матерњем језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразуми или погрешне интерпретације настале коришћењем овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
